# Urban Flood — v7: Seed-Blended Foundation (M1 NT1 + M1 NT2 + M2 NT2)


Key improvements over v5.5:

1. **Seed blending**: 5 seeds × CB+LGB = 10 models per submodel, averaged predictions

2. **Skip M2 NT1**: optimized separately, predictions merged later via `merge_m2nt1.py`

3. **Future rain features**: rain_future_1/3/5/10, future_sum5/10, future_max5/10, rain_remaining

4. **Batch prediction**: no WL lags / no AR (required for seed blending)

5. **Fixed 50/50 ensemble**: eliminates weight overfitting

6. All v5.5 features preserved: multi-hop graph, centrality, pipe capacity, enhanced rain, OOM fixes


M2 NT1 rows filled from best existing submission as placeholder.

In [1]:
from __future__ import annotations


import gc

import re

import time

import warnings

import ctypes

from collections import defaultdict

from pathlib import Path


import numpy as np

import pandas as pd

import pandas.api.types

from catboost import CatBoostRegressor

import lightgbm as lgb

from sklearn.model_selection import train_test_split


warnings.filterwarnings('ignore')

In [2]:
# =========================================================

# CONFIG (v7: seed blending, skip M2 NT1)

# =========================================================

DATA_ROOT = Path("/kaggle/input/datasets/bulbazavril/fullds/Models")


KEY_COLS = ["model_id", "event_id", "node_type", "node_id"]

TARGET_COL = "water_level"

SUBMISSION_COLS = ["row_id"] + KEY_COLS + [TARGET_COL]


VALID_EVENTS_PER_MODEL = 15

RANDOM_SEED = 42

MIN_PRED_TIMESTEP = 10

WARMUP_STEPS = 10

USE_TARGET_DELTA = False


# v7: Seed blending

SEED_LIST = [56, 228, 322]

SKIP_SUBMODELS = {(2, 1)}  # M2 NT1 optimized separately


CB_BASE = {

    'loss_function': 'RMSE',

    'eval_metric': 'RMSE',

    'task_type': 'GPU',

    'random_seed': 56,

    'verbose': 500,

}


CB_PARAMS = {

    (1, 1): {**CB_BASE, 'iterations': 10000, 'max_depth': 8, 'learning_rate': 0.05},

    (1, 2): {**CB_BASE, 'iterations': 5000, 'max_depth': 8, 'learning_rate': 0.15},

    (2, 2): {**CB_BASE, 'iterations': 5000, 'max_depth': 8, 'learning_rate': 0.15},

}


# --- LightGBM ---

LGB_BASE = {

    'objective': 'regression',

    'metric': 'rmse',

    'verbosity': -1,

    'n_jobs': -1,

    'random_state': 56,

    'device': 'gpu',

}


LGB_PARAMS = {

    (1, 1): {**LGB_BASE, 'n_estimators': 10000, 'max_depth': 8, 'learning_rate': 0.05,

             'reg_lambda': 1, 'subsample': 0.8, 'colsample_bytree': 0.8},

    (1, 2): {**LGB_BASE, 'n_estimators': 5000, 'max_depth': 8, 'learning_rate': 0.15,

             'reg_lambda': 1, 'subsample': 0.8, 'colsample_bytree': 0.8},

    (2, 2): {**LGB_BASE, 'n_estimators': 5000, 'max_depth': 8, 'learning_rate': 0.15,

             'reg_lambda': 3, 'subsample': 0.8, 'colsample_bytree': 0.8},

}

print("Config ready (v7: seed blending, skip M2 NT1).")

Config ready (v7: seed blending, skip M2 NT1).


In [3]:
# =========================================================

# METRIC

# =========================================================

STD_DEV_DICT = {

    (1, 1): 16.877747, (1, 2): 14.378797,

    (2, 1): 3.191784,  (2, 2): 2.727131,

}


def rmse(y_true, y_pred):

    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))


def local_score(solution_df, submission_df):

    sol = solution_df.copy()

    sub = submission_df.copy()

    model_scores = []

    for mid in sorted(sol['model_id'].unique()):

        nt_raw = {1: [], 2: []}

        event_scores = []

        for evid in sorted(sol[sol['model_id']==mid]['event_id'].unique()):

            nt_scores = []

            for nt in [1, 2]:

                mask_s = (sol['model_id']==mid) & (sol['event_id']==evid) & (sol['node_type']==nt)

                mask_b = (sub['model_id']==mid) & (sub['event_id']==evid) & (sub['node_type']==nt)

                if mask_s.sum() == 0: continue

                s_df = sol[mask_s]; b_df = sub[mask_b]

                sd = STD_DEV_DICT.get((mid, nt), 1.0)

                node_std_rmses = []

                for nid in s_df['node_id'].unique():

                    nm_s = s_df[s_df['node_id']==nid]['water_level'].values

                    nm_b = b_df[b_df['node_id']==nid]['water_level'].values

                    if len(nm_s) > 1 and len(nm_s) == len(nm_b):

                        r = rmse(nm_s, nm_b)

                        node_std_rmses.append(r / sd)

                        nt_raw[nt].append(r)

                if node_std_rmses:

                    nt_scores.append(np.mean(node_std_rmses))

            if nt_scores:

                event_scores.append(np.mean(nt_scores))

        if event_scores:

            ms = np.mean(event_scores)

            model_scores.append(ms)

            print(f"  Model {mid}: Std RMSE = {ms:.6f} ({len(event_scores)} events)")

            for nt in [1, 2]:

                if nt_raw[nt]:

                    print(f"    NT{nt}: raw RMSE mean={np.mean(nt_raw[nt]):.6f}")

    final = np.mean(model_scores)

    print(f"  Overall: {final:.6f}")

    return final

In [4]:
# =========================================================

# UTILITIES

# =========================================================

def _free_memory():

    gc.collect()

    try:

        ctypes.CDLL("libc.so.6").malloc_trim(0)

    except:

        pass


def ncol(col):

    col = str(col).strip().lower()

    col = col.replace('\u00b2', '2').replace('\u00b3', '3').replace('%', 'pct')

    col = re.sub(r'[^a-z0-9]+', '_', col)

    return re.sub(r'_+', '_', col).strip('_')


def read_csv(path):

    df = pd.read_csv(path)

    df.columns = [ncol(c) for c in df.columns]

    drop = [c for c in df.columns if c.startswith('unnamed')]

    if drop: df = df.drop(columns=drop)

    return df


def read_table(path):

    if str(path).endswith('.parquet'): df = pd.read_parquet(path)

    else: df = pd.read_csv(path)

    df.columns = [ncol(c) for c in df.columns]

    return df


def fcol(columns, names=(), tokens=(), required=True, label='col'):

    cols = list(columns)

    for n in names:

        if n in cols: return n

    for c in cols:

        if all(t in c for t in tokens): return c

    if required: raise KeyError(f"Cannot find {label} in {cols}")

    return None


def ensure_nid(df):

    c = fcol(df.columns, ('node_idx','node_id','id'), ('node',), False, 'nid')

    out = df.copy()

    if c is None: out['node_id'] = np.arange(len(out), dtype='int64')

    elif c != 'node_id': out = out.rename(columns={c: 'node_id'})

    out['node_id'] = out['node_id'].astype('int64')

    return out


def eid(d): return int(d.name.split('_')[1])

def mid_fn(d): return int(d.name.split('_')[1])

def event_dirs(split_dir):

    return sorted([d for d in split_dir.glob('event_*') if d.is_dir()], key=eid)


def dyn_path(event_dir, nt):

    return event_dir / f"{'1d' if nt==1 else '2d'}_nodes_dynamic_all.csv"


def edge_dyn_path(event_dir, nt):

    return event_dir / f"{'1d' if nt==1 else '2d'}_edges_dynamic_all.csv"


def find_file(model_dir, filename):

    for s in ('train', 'test'):

        p = model_dir / s / filename

        if p.exists(): return p

    return None


def downcast_float32(df):

    for c in df.columns:

        if df[c].dtype == 'float64':

            df[c] = df[c].astype('float32')

    return df


print("Utilities ready.")

Utilities ready.


In [5]:
# =========================================================

# STATIC NODE FEATURES

# =========================================================

STATIC_CACHE = {}


def load_static(model_id, node_type):

    k = (model_id, node_type)

    if k in STATIC_CACHE: return STATIC_CACHE[k]

    md = DATA_ROOT / f'Model_{model_id}'

    if node_type == 1:

        cands = ['1d_nodes_static.csv']

    else:

        cands = ['2d_nodes_static.csv', '2d_nodes_index.csv']

    for fn in cands:

        p = find_file(md, fn)

        if p:

            df = read_csv(p)

            df = ensure_nid(df)

            df = df.drop_duplicates('node_id').reset_index(drop=True)

            STATIC_CACHE[k] = df

            return df

    raise FileNotFoundError(f'Static not found for m{model_id} nt{node_type}')


print("Static loader ready.")

Static loader ready.


In [6]:
# =========================================================

# GRAPH FEATURES + MULTI-HOP + CENTRALITY + PIPE CAPACITY

# =========================================================

GRAPH_CACHE = {}

ADJ_CACHE = {}

EDGE_INDEX_CACHE = {}


def _count_upstream_total(upstream_dict, all_nids):

    cache = {}

    def _get_ancestors(nid, visited):

        if nid in cache: return cache[nid]

        if nid in visited: return set()

        visited.add(nid)

        result = set()

        for parent in upstream_dict.get(nid, []):

            result.add(parent)

            result.update(_get_ancestors(parent, visited))

        cache[nid] = result

        return result

    out = {}

    for nid in all_nids:

        out[int(nid)] = len(_get_ancestors(int(nid), set()))

    return out



def _compute_2hop_features(adj, node_feat_map, all_nids, feat_name):

    """Compute 2-hop neighbor aggregation for a given feature."""

    result = {}

    for nid in all_nids:

        nid = int(nid)

        hop1 = adj.get(nid, [])

        hop2_set = set()

        for n1 in hop1:

            for n2 in adj.get(n1, []):

                if n2 != nid and n2 not in hop1:

                    hop2_set.add(n2)

        if hop2_set:

            vals = [node_feat_map.get(n, np.nan) for n in hop2_set]

            vals = [v for v in vals if not np.isnan(v)]

            if vals:

                result[nid] = {

                    f'hop2_{feat_name}_mean': np.mean(vals),

                    f'hop2_{feat_name}_max': np.max(vals),

                    f'hop2_{feat_name}_min': np.min(vals),

                    f'hop2_count': len(hop2_set),

                }

    return result



def _pipe_capacity_manning(diameter, roughness, slope):

    """Manning's formula for full-pipe capacity: Q = (1/n) * A * R^(2/3) * S^(1/2).

    Circular pipe: A = pi*d^2/4, R = d/4.

    """

    if diameter <= 0 or roughness <= 0 or slope <= 0:

        return 0.0

    area = np.pi * diameter**2 / 4.0

    hyd_radius = diameter / 4.0

    return (1.0 / roughness) * area * hyd_radius**(2.0/3.0) * np.sqrt(slope)



def load_graph_feats(model_id, node_type):

    k = (model_id, node_type)

    if k in GRAPH_CACHE: return GRAPH_CACHE[k]

    md = DATA_ROOT / f'Model_{model_id}'

    prefix = '1d' if node_type == 1 else '2d'

    ei_path = find_file(md, f'{prefix}_edge_index.csv')


    if ei_path is None:

        print(f"  [WARN] {prefix}_edge_index.csv not found for Model_{model_id}")

        GRAPH_CACHE[k] = pd.DataFrame()

        return GRAPH_CACHE[k]


    ei = read_csv(ei_path)

    print(f"  Model {model_id} {prefix} edge_index: {len(ei)} edges, cols={list(ei.columns)}")


    cols = list(ei.columns)

    fc = fcol(cols, ('from_node','from_node_id','source'), ('from',), False, 'from')

    tc = fcol(cols, ('to_node','to_node_id','target'), ('to',), False, 'to')

    if fc is None or tc is None: fc, tc = cols[0], cols[1]


    EDGE_INDEX_CACHE[k] = (fc, tc, ei)


    adj = defaultdict(set)

    downstream = defaultdict(list)

    upstream = defaultdict(list)

    for _, r in ei.iterrows():

        u, v = int(r[fc]), int(r[tc])

        adj[u].add(v); adj[v].add(u)

        downstream[u].append(v)

        upstream[v].append(u)

    adj = {k_: list(v) for k_, v in adj.items()}

    ADJ_CACHE[k] = adj


    # Load edge static features

    es_path = find_file(md, f'{prefix}_edges_static.csv')

    edge_static = read_csv(es_path) if es_path else None

    if edge_static is not None:

        print(f"    Edge static: {len(edge_static)} edges, cols={list(edge_static.columns)}")


    efmap = {}

    if edge_static is not None and len(ei) == len(edge_static):

        for i in range(len(ei)):

            u, v = int(ei.iloc[i][fc]), int(ei.iloc[i][tc])

            efmap[(u, v)] = edge_static.iloc[i].to_dict()


    # 1D-2D connections

    conn = find_file(md, '1d2d_connections.csv')

    conn_map = defaultdict(list)

    conn_1d_to_2d = defaultdict(list)

    if conn:

        cdf = read_csv(conn)

        cc = list(cdf.columns)

        c1 = fcol(cc, ('1d_node_id','node_1d'), ('1d',), False, '1d')

        c2 = fcol(cc, ('2d_node_id','node_2d'), ('2d',), False, '2d')

        if c1 is None or c2 is None: c1, c2 = cc[0], cc[1]

        for _, r in cdf.iterrows():

            n1, n2 = int(r[c1]), int(r[c2])

            conn_1d_to_2d[n1].append(n2)

            if node_type == 1: conn_map[n1].append(n2)

            else: conn_map[n2].append(n1)


    static = load_static(model_id, node_type)

    all_nids = static['node_id'].unique()


    # Transitive upstream count (1D only)

    upstream_total = {}

    if node_type == 1:

        upstream_total = _count_upstream_total(upstream, all_nids)

        print(f"    Upstream total: max={max(upstream_total.values()) if upstream_total else 0}")


    # Elevation maps for gradients + 2-hop

    elev_map = {}

    if node_type == 2:

        elev_c = fcol(static.columns, ('elevation','min_elevation','centroid_elevation'),

                      ('elev',), False, 'elev')

        if elev_c:

            elev_map = dict(zip(static['node_id'].astype(int), static[elev_c].astype(float)))

    elif node_type == 1:

        inv_c = fcol(static.columns, ('invert_elevation',), ('invert',), False, 'inv')

        if inv_c:

            elev_map = dict(zip(static['node_id'].astype(int), static[inv_c].astype(float)))


    # Enhanced coupling: elevation difference 1D<->2D

    coupling_elev_diff = {}

    if node_type == 1 and conn_1d_to_2d:

        s2d = load_static(model_id, 2)

        elev_c_2d = fcol(s2d.columns, ('elevation','min_elevation','centroid_elevation'),

                         ('elev',), False, 'elev')

        inv_c = fcol(static.columns, ('invert_elevation',), ('invert',), False, 'inv')

        if elev_c_2d and inv_c:

            elev_2d_map = dict(zip(s2d['node_id'].astype(int), s2d[elev_c_2d].astype(float)))

            inv_map = dict(zip(static['node_id'].astype(int), static[inv_c].astype(float)))

            for n1d, n2d_list in conn_1d_to_2d.items():

                e2d_vals = [elev_2d_map.get(n2, np.nan) for n2 in n2d_list]

                e2d_clean = [v for v in e2d_vals if not np.isnan(v)]

                inv_val = inv_map.get(n1d, np.nan)

                if e2d_clean and not np.isnan(inv_val):

                    coupling_elev_diff[n1d] = np.mean(e2d_clean) - inv_val


    # v5.5: Graph centrality via NetworkX (fast for small 1D graphs, ok for 2D)

    centrality_pr = {}

    centrality_deg = {}

    try:

        import networkx as nx

        G = nx.Graph()

        for _, r in ei.iterrows():

            G.add_edge(int(r[fc]), int(r[tc]))

        centrality_pr = nx.pagerank(G, max_iter=100, tol=1e-4)

        centrality_deg = nx.degree_centrality(G)

        print(f"    Centrality: PageRank + degree for {G.number_of_nodes()} nodes")

    except Exception as e:

        print(f"    [WARN] centrality failed: {e}")


    # v5.5: 2-hop features

    hop2_feats = {}

    if elev_map:

        hop2_feats = _compute_2hop_features(adj, elev_map, all_nids, 'elev')

        print(f"    2-hop elev features: {len(hop2_feats)} nodes")


    # v5.5: Pipe capacity (1D only, Manning's formula)

    pipe_cap_map = {}

    if node_type == 1 and efmap:

        node_cap_in = defaultdict(list)

        node_cap_out = defaultdict(list)

        for (u, v), ef in efmap.items():

            diam = float(ef.get('diameter', 0) or 0)

            rough = float(ef.get('roughness', 0) or 0)

            slp = abs(float(ef.get('slope', 0) or 0))

            cap = _pipe_capacity_manning(diam, rough, slp)

            node_cap_out[u].append(cap)

            node_cap_in[v].append(cap)

        for nid in all_nids:

            nid = int(nid)

            caps_in = node_cap_in.get(nid, [])

            caps_out = node_cap_out.get(nid, [])

            all_caps = caps_in + caps_out

            if all_caps:

                pipe_cap_map[nid] = {

                    'pipe_cap_total_in': sum(caps_in),

                    'pipe_cap_total_out': sum(caps_out),

                    'pipe_cap_max': max(all_caps),

                    'pipe_cap_mean': np.mean(all_caps),

                }

        print(f"    Pipe capacity: {len(pipe_cap_map)} nodes")


    recs = []

    for nid in all_nids:

        nid = int(nid)

        r = {'node_id': nid, 'degree': len(adj.get(nid, []))}


        # Centrality

        r['pagerank'] = centrality_pr.get(nid, 0.0)

        r['degree_centrality'] = centrality_deg.get(nid, 0.0)


        # 2-hop features

        if nid in hop2_feats:

            r.update(hop2_feats[nid])


        if node_type == 1:

            up = upstream.get(nid, [])

            dn = downstream.get(nid, [])

            r['n_upstream'] = len(up)

            r['n_downstream'] = len(dn)

            r['upstream_total'] = upstream_total.get(nid, 0)


            # Aggregate upstream/downstream edge static features

            up_feats = [efmap.get((u, nid), {}) for u in up]

            dn_feats = [efmap.get((nid, v), {}) for v in dn]

            for pname, flist in [('up', up_feats), ('dn', dn_feats)]:

                if flist:

                    for fname in list(flist[0].keys())[:8]:

                        vals = []

                        for f in flist:

                            try: vals.append(float(f.get(fname, 0)))

                            except: pass

                        if vals:

                            r[f'{pname}_{fname}_mean'] = np.mean(vals)


            conn_nodes = conn_map.get(nid, [])

            r['has_2d_conn'] = int(len(conn_nodes) > 0)

            r['n_2d_conn'] = len(conn_nodes)

            r['coupling_elev_diff'] = coupling_elev_diff.get(nid, np.nan)


            # Pipe capacity

            if nid in pipe_cap_map:

                r.update(pipe_cap_map[nid])


        else:  # 2D

            conn_nodes = conn_map.get(nid, [])

            r['has_1d_conn'] = int(len(conn_nodes) > 0)

            r['n_1d_conn'] = len(conn_nodes)

            nbrs = adj.get(nid, [])

            if nbrs and elev_map:

                own_elev = elev_map.get(nid, np.nan)

                nbr_elevs = [elev_map.get(n, np.nan) for n in nbrs]

                nbr_elevs_clean = [e for e in nbr_elevs if not np.isnan(e)]

                if nbr_elevs_clean:

                    r['nbr_mean_elev'] = np.mean(nbr_elevs_clean)

                    r['nbr_min_elev'] = np.min(nbr_elevs_clean)

                    if not np.isnan(own_elev):

                        r['elev_gradient_mean'] = own_elev - np.mean(nbr_elevs_clean)

                        r['elev_gradient_max'] = own_elev - np.min(nbr_elevs_clean)


            # v5: aggregate 2D static edge features to nodes

            if efmap:

                nbr_edge_feats = []

                for nb in nbrs:

                    ef = efmap.get((nid, nb)) or efmap.get((nb, nid))

                    if ef: nbr_edge_feats.append(ef)

                if nbr_edge_feats:

                    for fname in list(nbr_edge_feats[0].keys())[:6]:

                        vals = []

                        for f in nbr_edge_feats:

                            try: vals.append(float(f.get(fname, 0)))

                            except: pass

                        if vals:

                            r[f'nbr_{fname}_mean'] = np.mean(vals)

        recs.append(r)


    result = pd.DataFrame(recs)

    result['node_id'] = result['node_id'].astype('int64')

    print(f"    Graph features: {len(result)} nodes, {len(result.columns)-1} features")

    GRAPH_CACHE[k] = result

    return result


print("Graph feature loader ready (v5.5: +multi-hop, +centrality, +pipe capacity).")

Graph feature loader ready (v5.5: +multi-hop, +centrality, +pipe capacity).


In [7]:
# =========================================================

# DIST TO NEAREST DRAIN (for 2D nodes)

# =========================================================

DIST_DRAIN_CACHE = {}


def load_dist_to_drain(model_id):

    if model_id in DIST_DRAIN_CACHE: return DIST_DRAIN_CACHE[model_id]

    try:

        s1d = load_static(model_id, 1)

        s2d = load_static(model_id, 2)

        x1 = fcol(s1d.columns, ('position_x',), ('position','x'), False, 'x')

        y1 = fcol(s1d.columns, ('position_y',), ('position','y'), False, 'y')

        x2 = fcol(s2d.columns, ('position_x',), ('position','x'), False, 'x')

        y2 = fcol(s2d.columns, ('position_y',), ('position','y'), False, 'y')

        if not all([x1, y1, x2, y2]):

            DIST_DRAIN_CACHE[model_id] = pd.DataFrame()

            return DIST_DRAIN_CACHE[model_id]

        from scipy.spatial import cKDTree

        tree = cKDTree(s1d[[x1, y1]].values.astype(float))

        dists, _ = tree.query(s2d[[x2, y2]].values.astype(float), k=min(3, len(s1d)))

        result = pd.DataFrame({'node_id': s2d['node_id'].values,

                               'dist_nearest_drain': dists[:, 0] if dists.ndim > 1 else dists})

        if dists.ndim > 1 and dists.shape[1] >= 2: result['dist_2nd_drain'] = dists[:, 1]

        if dists.ndim > 1 and dists.shape[1] >= 3: result['dist_3rd_drain'] = dists[:, 2]

        result['node_id'] = result['node_id'].astype('int64')

        print(f"  dist_nearest_drain Model_{model_id}: {len(result)} nodes")

        DIST_DRAIN_CACHE[model_id] = result

        return result

    except Exception as e:

        print(f"  [WARN] dist_to_drain failed: {e}")

        DIST_DRAIN_CACHE[model_id] = pd.DataFrame()

        return DIST_DRAIN_CACHE[model_id]


print("Dist-to-drain ready.")

Dist-to-drain ready.


In [8]:
# =========================================================

# RAIN FEATURES (v5.5: +duration, +dry spell, +quantiles)

# =========================================================

RAIN_CACHE = {}

TIMESTEP_DURATION_CACHE = {}


def load_timestep_durations(model_id, event_dir):

    """v5.5: Load actual timestep durations from timesteps.csv."""

    k = (model_id, eid(event_dir))

    if k in TIMESTEP_DURATION_CACHE: return TIMESTEP_DURATION_CACHE[k]

    ts_path = event_dir / 'timesteps.csv'

    if not ts_path.exists():

        TIMESTEP_DURATION_CACHE[k] = None

        return None

    try:

        ts_df = read_csv(ts_path)

        # Expect columns like: timestep, datetime or time or seconds

        ts_col = fcol(ts_df.columns, ('timestep',), ('time','step'), False, 'ts')

        time_col = fcol(ts_df.columns, ('datetime','time','seconds','elapsed'),

                        ('time',), False, 'time')

        if ts_col is None or time_col is None:

            TIMESTEP_DURATION_CACHE[k] = None

            return None

        ts_df = ts_df.sort_values(ts_col).reset_index(drop=True)

        # Try to parse as seconds or datetime

        try:

            time_vals = pd.to_numeric(ts_df[time_col])

            durations = time_vals.diff().fillna(time_vals.iloc[0] if len(time_vals) > 0 else 0)

        except (ValueError, TypeError):

            try:

                time_vals = pd.to_datetime(ts_df[time_col])

                durations = time_vals.diff().dt.total_seconds().fillna(0)

            except:

                TIMESTEP_DURATION_CACHE[k] = None

                return None

        result = pd.DataFrame({

            'timestep': ts_df[ts_col].astype('int64'),

            'step_duration': durations.astype('float32'),

            'elapsed_time': time_vals.astype('float64') if pd.api.types.is_numeric_dtype(time_vals) else durations.cumsum().astype('float32'),

        })

        TIMESTEP_DURATION_CACHE[k] = result

        return result

    except Exception:

        TIMESTEP_DURATION_CACHE[k] = None

        return None



def load_rain(model_id, event_dir):

    k = (model_id, eid(event_dir))

    if k in RAIN_CACHE: return RAIN_CACHE[k]

    path_2d = dyn_path(event_dir, 2)

    df = read_csv(path_2d)

    df = ensure_nid(df)

    ts_col = fcol(df.columns, ('timestep',), ('time','step'), label='timestep')

    rain_col = fcol(df.columns, ('rainfall',), ('rain',), label='rainfall')

    rain_ts = df.groupby(ts_col)[rain_col].mean().reset_index()

    rain_ts.columns = ['timestep', 'rain_global']

    rain_ts = rain_ts.sort_values('timestep').reset_index(drop=True)

    rain = rain_ts['rain_global'].astype('float64')


    out = pd.DataFrame({

        'timestep': rain_ts['timestep'].astype('int64'),

        'rain_global': rain,

        'rain_lag1': rain.shift(1).fillna(0),

        'rain_lag2': rain.shift(2).fillna(0),

        'rain_lag3': rain.shift(3).fillna(0),

        'rain_roll3': rain.rolling(3, min_periods=1).sum(),

        'rain_roll6': rain.rolling(6, min_periods=1).sum(),

        'rain_roll12': rain.rolling(12, min_periods=1).sum(),

        'rain_roll24': rain.rolling(24, min_periods=1).sum(),

        'rain_cumsum': rain.cumsum(),

        'rain_delta': rain - rain.shift(1).fillna(0),

        'is_raining': (rain > 0).astype('int8'),

        'rain_intensity_peak': rain.expanding().max(),

    })


    out['rain_accel'] = out['rain_delta'] - out['rain_delta'].shift(1).fillna(0)

    for alpha in [0.1, 0.05, 0.02, 0.01]:

        out[f'rain_ema_{int(1/alpha)}'] = rain.ewm(alpha=alpha, adjust=False).mean()


    total = rain.sum()

    out['rain_cum_ratio'] = (rain.cumsum() / total) if total > 0 else 0


    peak = rain.max()

    peak_ts = rain_ts.loc[rain.idxmax(), 'timestep'] if peak > 0 else 0

    max_ts = rain_ts['timestep'].max()

    out['event_total_rain'] = total

    out['event_peak_rain'] = peak

    out['time_since_rain_peak'] = out['timestep'] - peak_ts

    out['frac_event'] = out['timestep'] / (max_ts + 1)

    out['rain_phase_rising'] = (out['rain_delta'] > 0).astype('int8')

    out['rain_phase_falling'] = ((out['rain_delta'] < 0) & (rain > 0)).astype('int8')


    # v5.5: Rain duration & dry spell

    is_rain = (rain > 0).values

    rain_dur = np.zeros(len(rain), dtype='int32')

    dry_spell = np.zeros(len(rain), dtype='int32')

    cnt_rain, cnt_dry = 0, 0

    for i in range(len(rain)):

        if is_rain[i]:

            cnt_rain += 1

            cnt_dry = 0

        else:

            cnt_dry += 1

            cnt_rain = 0

        rain_dur[i] = cnt_rain

        dry_spell[i] = cnt_dry

    out['rain_duration'] = rain_dur

    out['dry_spell'] = dry_spell


    # v5.5: Intensity quantiles (expanding)

    out['rain_q75'] = rain.expanding().quantile(0.75)

    out['rain_q90'] = rain.expanding().quantile(0.90)


    # v5.5: Rain rate of change (smoothed)

    out['rain_roc_smooth'] = rain.rolling(3, min_periods=1).mean().diff().fillna(0)


    # v5.5: Timestep durations

    ts_dur = load_timestep_durations(model_id, event_dir)

    if ts_dur is not None:

        out = out.merge(ts_dur, on='timestep', how='left')


    RAIN_CACHE[k] = out

    return out


print("Rain features ready (v5.5: +duration, +dry_spell, +quantiles, +timestep_duration).")

Rain features ready (v5.5: +duration, +dry_spell, +quantiles, +timestep_duration).


In [9]:
# =========================================================

# WARMUP FEATURES + TRENDS + NEIGHBOR WARMUP

# + v5 NEW: INLET FLOW, WATER VOLUME, EDGE DYNAMICS

# =========================================================

WARMUP_CACHE = {}

WARMUP_DYN_CACHE = {}


def load_warmup(model_id, split, event_dir, node_type):

    key = (model_id, eid(event_dir), node_type, split)

    if key in WARMUP_CACHE: return WARMUP_CACHE[key]

    df = read_csv(dyn_path(event_dir, node_type))

    df = ensure_nid(df)

    ts_col = fcol(df.columns, ('timestep',), ('time','step'), label='timestep')

    wl_col = fcol(df.columns, ('water_level',), ('water','level'), label='water_level')

    warm = df[['node_id', ts_col, wl_col]].copy()

    warm.columns = ['node_id', 'timestep', 'water_level']

    warm['timestep'] = warm['timestep'].astype('int64')

    warm['water_level'] = warm['water_level'].astype('float64')

    warm = warm.sort_values(['node_id', 'timestep']).reset_index(drop=True)

    warm = warm[warm['timestep'] < WARMUP_STEPS].copy()

    warm['wi'] = warm.groupby('node_id').cumcount()

    wide = warm.pivot(index='node_id', columns='wi', values='water_level')

    wide.columns = [f'water_level_{int(c)}' for c in wide.columns]

    wide = wide.reset_index()

    for i in range(WARMUP_STEPS):

        c = f'water_level_{i}'

        if c not in wide.columns: wide[c] = np.nan

    WARMUP_CACHE[key] = wide

    return wide



def _agg_warmup_series(vals):

    clean = vals[~np.isnan(vals)]

    if len(clean) == 0:

        return np.nan, np.nan, np.nan, np.nan

    return np.mean(clean), np.max(clean), np.min(clean), clean[-1]



def load_warmup_dynamics(model_id, split, event_dir, node_type):

    key = (model_id, eid(event_dir), node_type, split)

    if key in WARMUP_DYN_CACHE: return WARMUP_DYN_CACHE[key]


    result_recs = {}


    # --- Node-level dynamic features (inlet_flow / water_volume) ---

    node_dyn = read_csv(dyn_path(event_dir, node_type))

    node_dyn = ensure_nid(node_dyn)

    ts_col = fcol(node_dyn.columns, ('timestep',), ('time','step'), label='timestep')

    node_dyn['timestep'] = node_dyn[ts_col].astype('int64')

    warmup_nodes = node_dyn[node_dyn['timestep'] < WARMUP_STEPS].copy()


    if node_type == 1:

        inlet_col = fcol(warmup_nodes.columns, ('1d_node_inlet_flow','inlet_flow'),

                         ('inlet','flow'), False, 'inlet_flow')

        if inlet_col:

            for nid, grp in warmup_nodes.groupby('node_id'):

                vals = grp[inlet_col].astype('float64').values

                mn, mx, mi, last = _agg_warmup_series(vals)

                result_recs.setdefault(int(nid), {})['inlet_flow_mean'] = mn

                result_recs.setdefault(int(nid), {})['inlet_flow_max'] = mx

                result_recs.setdefault(int(nid), {})['inlet_flow_min'] = mi

                result_recs.setdefault(int(nid), {})['inlet_flow_last'] = last

                abs_vals = np.abs(vals[~np.isnan(vals)])

                result_recs[int(nid)]['inlet_flow_abs_mean'] = np.mean(abs_vals) if len(abs_vals) else np.nan

                result_recs[int(nid)]['inlet_flow_abs_max'] = np.max(abs_vals) if len(abs_vals) else np.nan

    else:

        vol_col = fcol(warmup_nodes.columns, ('water_volume',), ('volume',), False, 'volume')

        if vol_col:

            for nid, grp in warmup_nodes.groupby('node_id'):

                vals = grp[vol_col].astype('float64').values

                mn, mx, mi, last = _agg_warmup_series(vals)

                result_recs.setdefault(int(nid), {})['wvol_mean'] = mn

                result_recs.setdefault(int(nid), {})['wvol_max'] = mx

                result_recs.setdefault(int(nid), {})['wvol_last'] = last

                clean = vals[~np.isnan(vals)]

                if len(clean) >= 2:

                    result_recs[int(nid)]['wvol_trend'] = clean[-1] - clean[0]

                else:

                    result_recs[int(nid)]['wvol_trend'] = np.nan


    del warmup_nodes, node_dyn


    # --- Edge-level dynamic features aggregated to nodes ---

    edge_path = edge_dyn_path(event_dir, node_type)

    k_ei = (model_id, node_type)

    if edge_path.exists() and k_ei in EDGE_INDEX_CACHE:

        fc_name, tc_name, ei_df = EDGE_INDEX_CACHE[k_ei]

        edge_dyn = read_csv(edge_path)

        ts_e = fcol(edge_dyn.columns, ('timestep',), ('time','step'), label='timestep')

        edge_dyn['timestep'] = edge_dyn[ts_e].astype('int64')

        warmup_edges = edge_dyn[edge_dyn['timestep'] < WARMUP_STEPS].copy()

        del edge_dyn


        if node_type == 1:

            flow_col = fcol(warmup_edges.columns, ('1d_edge_flow','edge_flow'),

                            ('edge','flow'), False, 'flow')

            vel_col = fcol(warmup_edges.columns, ('1d_edge_velocity','edge_velocity'),

                           ('velocity',), False, 'velocity')

        else:

            flow_col = fcol(warmup_edges.columns, ('2d_flow',), ('flow',), False, 'flow')

            vel_col = fcol(warmup_edges.columns, ('2d_velocity',), ('velocity',), False, 'velocity')


        if flow_col or vel_col:

            eidx_col = fcol(warmup_edges.columns, ('edge_idx','edge_id','link_id'),

                            ('edge',), False, 'edge_idx')


            edge_to_nodes = {}

            for i in range(len(ei_df)):

                edge_to_nodes[i] = (int(ei_df.iloc[i][fc_name]), int(ei_df.iloc[i][tc_name]))


            node_in_flows, node_out_flows = defaultdict(list), defaultdict(list)

            node_in_vels, node_out_vels = defaultdict(list), defaultdict(list)


            if eidx_col:

                edge_groups = warmup_edges.groupby(eidx_col)

            else:

                n_edges = len(ei_df)

                warmup_edges['_edge_idx'] = warmup_edges.groupby('timestep').cumcount()

                edge_groups = warmup_edges.groupby('_edge_idx')

                eidx_col = '_edge_idx'


            for edge_idx, grp in edge_groups:

                edge_idx = int(edge_idx)

                if edge_idx not in edge_to_nodes: continue

                from_n, to_n = edge_to_nodes[edge_idx]


                if flow_col and flow_col in grp.columns:

                    fvals = grp[flow_col].astype('float64').values

                    fmean = np.nanmean(fvals)

                    fabs_mean = np.nanmean(np.abs(fvals))

                    fabs_max = np.nanmax(np.abs(fvals))

                    node_out_flows[from_n].append((fmean, fabs_mean, fabs_max))

                    node_in_flows[to_n].append((fmean, fabs_mean, fabs_max))


                if vel_col and vel_col in grp.columns:

                    vvals = grp[vel_col].astype('float64').values

                    vmean = np.nanmean(vvals)

                    vabs_mean = np.nanmean(np.abs(vvals))

                    vabs_max = np.nanmax(np.abs(vvals))

                    node_out_vels[from_n].append((vmean, vabs_mean, vabs_max))

                    node_in_vels[to_n].append((vmean, vabs_mean, vabs_max))


            del warmup_edges


            prefix = '1d' if node_type == 1 else '2d'

            all_nids = set()

            for d in [node_in_flows, node_out_flows, node_in_vels, node_out_vels]:

                all_nids.update(d.keys())


            for nid in all_nids:

                rec = result_recs.setdefault(int(nid), {})


                if nid in node_in_flows:

                    in_f = node_in_flows[nid]

                    rec[f'{prefix}_in_flow_mean'] = np.mean([x[0] for x in in_f])

                    rec[f'{prefix}_in_flow_abs_mean'] = np.mean([x[1] for x in in_f])

                    rec[f'{prefix}_in_flow_abs_max'] = np.max([x[2] for x in in_f])


                if nid in node_out_flows:

                    out_f = node_out_flows[nid]

                    rec[f'{prefix}_out_flow_mean'] = np.mean([x[0] for x in out_f])

                    rec[f'{prefix}_out_flow_abs_mean'] = np.mean([x[1] for x in out_f])

                    rec[f'{prefix}_out_flow_abs_max'] = np.max([x[2] for x in out_f])


                if nid in node_in_vels:

                    in_v = node_in_vels[nid]

                    rec[f'{prefix}_in_vel_mean'] = np.mean([x[0] for x in in_v])

                    rec[f'{prefix}_in_vel_abs_mean'] = np.mean([x[1] for x in in_v])

                    rec[f'{prefix}_in_vel_abs_max'] = np.max([x[2] for x in in_v])


                if nid in node_out_vels:

                    out_v = node_out_vels[nid]

                    rec[f'{prefix}_out_vel_mean'] = np.mean([x[0] for x in out_v])

                    rec[f'{prefix}_out_vel_abs_mean'] = np.mean([x[1] for x in out_v])

                    rec[f'{prefix}_out_vel_abs_max'] = np.max([x[2] for x in out_v])


                if node_type == 1:

                    in_mean = rec.get(f'{prefix}_in_flow_mean', 0) or 0

                    out_mean = rec.get(f'{prefix}_out_flow_mean', 0) or 0

                    rec[f'{prefix}_net_flow'] = in_mean - out_mean


    if result_recs:

        rows = [{'node_id': nid, **feats} for nid, feats in result_recs.items()]

        result = pd.DataFrame(rows)

        result['node_id'] = result['node_id'].astype('int64')

    else:

        result = pd.DataFrame()


    WARMUP_DYN_CACHE[key] = result

    return result



def compute_warmup_trends(warmup_wide, static_df, node_type):

    wl_cols = sorted([c for c in warmup_wide.columns if c.startswith('water_level_')])

    trend = pd.DataFrame({'node_id': warmup_wide['node_id'].astype('int64')})

    if not wl_cols: return trend

    vals = warmup_wide[wl_cols].values.astype('float64')

    trend['warmup_mean'] = np.nanmean(vals, axis=1)

    trend['warmup_std'] = np.nanstd(vals, axis=1)

    trend['warmup_range'] = np.nanmax(vals, axis=1) - np.nanmin(vals, axis=1)

    trend['warmup_last'] = vals[:, -1]

    if vals.shape[1] >= 2:

        trend['warmup_last_delta'] = vals[:, -1] - vals[:, -2]

    if vals.shape[1] >= 3:

        trend['warmup_accel'] = (vals[:, -1] - vals[:, -2]) - (vals[:, -2] - vals[:, -3])

    x = np.arange(vals.shape[1], dtype='float64')

    xm = x.mean(); xvar = ((x - xm)**2).sum()

    if xvar > 0:

        trend['warmup_trend'] = np.nansum((x[None,:] - xm) * vals, axis=1) / xvar

    if node_type == 1:

        inv_c = fcol(static_df.columns, ('invert_elevation',), ('invert',), False, 'inv')

        surf_c = fcol(static_df.columns, ('surface_elevation',), ('surface',), False, 'surf')

        if inv_c and surf_c:

            merged = trend.merge(static_df[['node_id', inv_c, surf_c]], on='node_id', how='left')

            inv_v = merged[inv_c].fillna(0).values

            surf_v = merged[surf_c].fillna(0).values

            pipe_d = surf_v - inv_v

            trend['fill_ratio'] = np.where(pipe_d > 0, (trend['warmup_last'].values - inv_v) / pipe_d, 0)

            trend['is_surcharged'] = (trend['warmup_last'].values > surf_v).astype('int8')

    return trend



def compute_neighbor_warmup(model_id, node_type, warmup_wide):

    adj = ADJ_CACHE.get((model_id, node_type), {})

    if not adj or 'water_level_9' not in warmup_wide.columns:

        return pd.DataFrame()

    wl9_map = dict(zip(warmup_wide['node_id'].astype(int),

                       warmup_wide['water_level_9'].astype('float64')))

    records = []

    for nid in warmup_wide['node_id'].astype(int):

        nid = int(nid)

        nbrs = adj.get(nid, [])

        own_wl9 = wl9_map.get(nid, np.nan)

        if nbrs:

            nbr_wl9 = [wl9_map.get(n, np.nan) for n in nbrs]

            nbr_mean = np.nanmean(nbr_wl9)

            records.append({'node_id': nid, 'nbr_mean_wl9': nbr_mean,

                           'nbr_max_wl9': np.nanmax(nbr_wl9),

                           'gradient_wl9': (own_wl9 - nbr_mean) if pd.notna(own_wl9) and pd.notna(nbr_mean) else np.nan})

        else:

            records.append({'node_id': nid, 'nbr_mean_wl9': np.nan,

                           'nbr_max_wl9': np.nan, 'gradient_wl9': np.nan})

    return pd.DataFrame(records)


print("Warmup features ready (v5: +inlet_flow, +water_volume, +edge dynamics).")

Warmup features ready (v5: +inlet_flow, +water_volume, +edge dynamics).


In [10]:
# =========================================================

# BUILD SUPERVISED FRAME (v7: batch prediction, no WL lags)

# =========================================================

EVENT_FRAME_CACHE = {}


def build_supervised_event_frame(model_id, split, event_dir, node_type, include_target=True):

    cache_key = (model_id, eid(event_dir), node_type, split, include_target)

    if cache_key in EVENT_FRAME_CACHE:

        return EVENT_FRAME_CACHE[cache_key].copy()


    df_dyn = read_csv(dyn_path(event_dir, node_type))

    df_dyn = ensure_nid(df_dyn)

    ts_col = fcol(df_dyn.columns, ('timestep',), ('time','step'), label='timestep')

    wl_col = fcol(df_dyn.columns, ('water_level',), ('water','level'),

                  required=include_target, label='water_level')

    rain_col = fcol(df_dyn.columns, ('rainfall',), ('rain',), required=False, label='rainfall')


    keep = [ts_col, 'node_id']

    if include_target and wl_col: keep.append(wl_col)

    if node_type == 2 and rain_col: keep.append(rain_col)


    df = df_dyn[keep].copy()

    del df_dyn

    renames = {ts_col: 'timestep'}

    if include_target and wl_col: renames[wl_col] = 'water_level'

    if node_type == 2 and rain_col: renames[rain_col] = 'rain_local'

    df = df.rename(columns=renames)

    df['timestep'] = df['timestep'].astype('int64')

    df = df[df['timestep'] >= MIN_PRED_TIMESTEP].copy()


    df['model_id'] = model_id

    df['event_id'] = eid(event_dir)

    df['node_type'] = node_type


    # Static features

    static = load_static(model_id, node_type)

    df = df.merge(static, on='node_id', how='left')


    # Rain features

    rain = load_rain(model_id, event_dir)

    df = df.merge(rain, on='timestep', how='left')


    # Warmup water levels

    warmup = load_warmup(model_id, split, event_dir, node_type)

    df = df.merge(warmup, on='node_id', how='left')


    # Graph features

    gf = load_graph_feats(model_id, node_type)

    if len(gf) > 0 and 'node_id' in gf.columns:

        df = df.merge(gf, on='node_id', how='left')


    # Distance to drain (2D)

    if node_type == 2:

        dd = load_dist_to_drain(model_id)

        if len(dd) > 0 and 'node_id' in dd.columns:

            df = df.merge(dd, on='node_id', how='left')


    # Warmup trends

    trend = compute_warmup_trends(warmup, static, node_type)

    df = df.merge(trend, on='node_id', how='left')


    # Neighbor warmup

    nbr = compute_neighbor_warmup(model_id, node_type, warmup)

    if len(nbr) > 0:

        df = df.merge(nbr, on='node_id', how='left')


    # Warmup dynamics (inlet_flow, water_volume, edge flow/velocity)

    wdyn = load_warmup_dynamics(model_id, split, event_dir, node_type)

    if len(wdyn) > 0 and 'node_id' in wdyn.columns:

        df = df.merge(wdyn, on='node_id', how='left')


    if 'rain_local' not in df.columns:

        df['rain_local'] = df.get('rain_global', 0)


    # Interaction features (1D only)

    if 'upstream_total' in df.columns:

        df['upstream_x_rain_cumsum'] = df['upstream_total'] * df['rain_cumsum']

        df['upstream_x_rain_ema_50'] = df['upstream_total'] * df.get('rain_ema_50', 0)


    if 'inlet_flow_abs_mean' in df.columns:

        df['inlet_x_rain_cumsum'] = df['inlet_flow_abs_mean'] * df['rain_cumsum']


    if 'fill_ratio' in df.columns:

        df['fill_x_rain_cumsum'] = df['fill_ratio'] * df['rain_cumsum']


    if 'pipe_cap_total_in' in df.columns:

        df['cap_x_rain_cumsum'] = df['pipe_cap_total_in'] * df['rain_cumsum']

        df['cap_x_fill'] = df['pipe_cap_total_in'] * df.get('fill_ratio', 0)




    df['node_id_cat'] = df['node_id'].astype(str)

    df['log_timestep'] = np.log1p(df['timestep'].astype('float32'))


    if include_target and USE_TARGET_DELTA and 'water_level_9' in df.columns:

        df['wl_delta_target'] = df['water_level'].astype('float64') - df['water_level_9'].astype('float64')


    for c in df.columns:

        if c == 'node_id_cat': continue

        if df[c].dtype == 'object':

            conv = pd.to_numeric(df[c], errors='coerce')

            if conv.notna().mean() >= 0.95: df[c] = conv

            else: df[c] = df[c].fillna('NA').astype(str)


    df = df.sort_values(['node_id', 'timestep']).reset_index(drop=True)

    if include_target:

        df['water_level'] = df['water_level'].astype('float64')


    # Downcast to float32 (keep target as float64)

    for c in df.columns:

        if c in (TARGET_COL, 'wl_delta_target'): continue

        if df[c].dtype == 'float64':

            df[c] = df[c].astype('float32')


    EVENT_FRAME_CACHE[cache_key] = df.copy()

    return df



def get_features(df):

    excluded = set(KEY_COLS + [TARGET_COL, 'event_id', 'wl_delta_target'])

    feat = [c for c in df.columns if c not in excluded]

    cat = [c for c in feat if df[c].dtype == 'object' or c.endswith('_cat')]

    return feat, cat





print("Frame builder ready (v7: batch, +future rain, no WL lags).")

Frame builder ready (v7: batch, +future rain, no WL lags).


In [11]:
# =========================================================

# DATASET INDEXING & SPLIT

# =========================================================

assert DATA_ROOT.exists()

model_dirs = sorted(DATA_ROOT.glob('Model_*'))

assert len(model_dirs) == 2


train_events, test_events = {}, {}

for d in model_dirs:

    m = mid_fn(d)

    train_events[m] = event_dirs(d / 'train')

    test_events[m] = event_dirs(d / 'test')

    print(f"Model {m}: {len(train_events[m])} train, {len(test_events[m])} test")


train_pool, valid_pool = {}, {}

for m in sorted(train_events):

    tr, va = train_test_split(train_events[m], test_size=VALID_EVENTS_PER_MODEL,

                               random_state=RANDOM_SEED + m, shuffle=True)

    train_pool[m] = sorted(tr, key=eid)

    valid_pool[m] = sorted(va, key=eid)

    print(f"  Split: {len(train_pool[m])} train, {len(valid_pool[m])} valid")

Model 1: 68 train, 29 test
Model 2: 69 train, 30 test
  Split: 53 train, 15 valid
  Split: 54 train, 15 valid


In [12]:
# =========================================================

# INSPECT FEATURES

# =========================================================

for nt in (1, 2):

    sf = build_supervised_event_frame(1, 'train', train_pool[1][0], nt)

    feats, cats = get_features(sf)

    print(f"\n{'1D' if nt==1 else '2D'} Features: {len(feats)}")

    for i, f in enumerate(feats):

        print(f"  {i:3d}. {f}")

    del sf

gc.collect()

print("Done.")

  Model 1 1d edge_index: 16 edges, cols=['edge_idx', 'from_node', 'to_node']
    Edge static: 16 edges, cols=['edge_idx', 'relative_position_x', 'relative_position_y', 'length', 'diameter', 'shape', 'roughness', 'slope']
    Upstream total: max=16
    Centrality: PageRank + degree for 17 nodes
    2-hop elev features: 17 nodes
    Pipe capacity: 17 nodes
    Graph features: 17 nodes, 33 features

1D Features: 115
    0. timestep
    1. position_x
    2. position_y
    3. depth
    4. invert_elevation
    5. surface_elevation
    6. base_area
    7. rain_global
    8. rain_lag1
    9. rain_lag2
   10. rain_lag3
   11. rain_roll3
   12. rain_roll6
   13. rain_roll12
   14. rain_roll24
   15. rain_cumsum
   16. rain_delta
   17. is_raining
   18. rain_intensity_peak
   19. rain_accel
   20. rain_ema_10
   21. rain_ema_20
   22. rain_ema_50
   23. rain_ema_100
   24. rain_cum_ratio
   25. event_total_rain
   26. event_peak_rain
   27. time_since_rain_peak
   28. frac_event
   29. rain_phas

In [13]:
# =========================================================

# TRAIN: CatBoost + LightGBM (v7: skip M2 NT1, no noise)

# =========================================================

cb_models, lgb_models = {}, {}

specs = {}

sol_parts = []

sub_parts_cb, sub_parts_lgb = [], []

valid_preds = {}


for m in sorted(train_pool):

    for nt in (1, 2):

        if (m, nt) in SKIP_SUBMODELS:

            print(f"\n  Skipping M{m} NT{nt} (optimized separately)")

            continue


        print(f"\n{'='*60}")

        print(f"=== Model {m}, NodeType {nt} ===")


        # v5.5 OOM FIX: clear caches before building frames for this submodel

        EVENT_FRAME_CACHE.clear()

        RAIN_CACHE.clear()

        WARMUP_CACHE.clear()

        WARMUP_DYN_CACHE.clear()

        _free_memory()


        cb_params = CB_PARAMS.get((m, nt), {**CB_BASE, 'iterations': 3000, 'max_depth': 8})

        lgb_params = LGB_PARAMS.get((m, nt), {**LGB_BASE, 'n_estimators': 3000, 'max_depth': 8})


        train_frames = [build_supervised_event_frame(m, 'train', ed, nt) for ed in train_pool[m]]

        valid_frames = [build_supervised_event_frame(m, 'train', ed, nt) for ed in valid_pool[m]]


        # Clear cache immediately after building frames

        EVENT_FRAME_CACHE.clear()

        WARMUP_CACHE.clear()

        WARMUP_DYN_CACHE.clear()

        _free_memory()


        train_df = pd.concat(train_frames, ignore_index=True)

        del train_frames

        valid_df = pd.concat(valid_frames, ignore_index=True)

        del valid_frames

        _free_memory()


        feat, cat = get_features(train_df)

        use_delta = USE_TARGET_DELTA and 'wl_delta_target' in train_df.columns

        target = 'wl_delta_target' if use_delta else TARGET_COL

        print(f"  Features: {len(feat)}, Rows: train={len(train_df):,} valid={len(valid_df):,}")

        print(f"  Target: {target}")



        # --- CatBoost ---

        print(f"\n  --- CatBoost ---")

        cb = CatBoostRegressor(**cb_params)

        cb.fit(train_df[feat], train_df[target],

               eval_set=(valid_df[feat], valid_df[target]),

               cat_features=cat, verbose=cb_params.get('verbose', 500))


        pred_cb = cb.predict(valid_df[feat])

        if use_delta: pred_cb = pred_cb + valid_df['water_level_9'].values


        cb_rmse = np.sqrt(np.mean((valid_df[TARGET_COL].values - pred_cb)**2))

        sd = STD_DEV_DICT.get((m, nt), 1.0)

        print(f"  CB Raw RMSE: {cb_rmse:.6f}, Std: {cb_rmse/sd:.6f}")


        best_iter_cb = cb.get_best_iteration()

        if best_iter_cb is None: best_iter_cb = cb.tree_count_

        print(f"  CB best iter: {best_iter_cb}")


        fi = cb.get_feature_importance(prettified=True)

        print(f"  CB Top-10:")

        print(fi.head(10).to_string(index=False))


        # --- LightGBM ---

        print(f"\n  --- LightGBM ---")

        feat_lgb = [f for f in feat if f != 'node_id_cat']


        lgb_model = lgb.LGBMRegressor(**lgb_params)

        lgb_model.fit(

            train_df[feat_lgb], train_df[target],

            eval_set=[(valid_df[feat_lgb], valid_df[target])],

            callbacks=[lgb.early_stopping(300), lgb.log_evaluation(300)],

        )


        pred_lgb_v = lgb_model.predict(valid_df[feat_lgb])

        if use_delta: pred_lgb_v = pred_lgb_v + valid_df['water_level_9'].values


        lgb_rmse = np.sqrt(np.mean((valid_df[TARGET_COL].values - pred_lgb_v)**2))

        print(f"  LGB Raw RMSE: {lgb_rmse:.6f}, Std: {lgb_rmse/sd:.6f}")

        print(f"  LGB best iter: {lgb_model.best_iteration_}")


        # Store validation predictions

        valid_preds[(m, nt)] = (

            valid_df[TARGET_COL].values.copy(),

            pred_cb.copy(),

            pred_lgb_v.copy(),

        )


        sol_parts.append(valid_df[KEY_COLS + [TARGET_COL]].copy())

        sub_parts_cb.append(valid_df[KEY_COLS].copy().assign(**{TARGET_COL: pred_cb}))

        sub_parts_lgb.append(valid_df[KEY_COLS].copy().assign(**{TARGET_COL: pred_lgb_v}))


        cb_models[(m, nt)] = cb

        lgb_models[(m, nt)] = lgb_model

        specs[(m, nt)] = {'feat': feat, 'feat_lgb': feat_lgb, 'cat': cat,

                          'use_delta': use_delta,

                          'best_iter_cb': best_iter_cb,

                          'best_iter_lgb': lgb_model.best_iteration_}


        del train_df, valid_df

        _free_memory()


=== Model 1, NodeType 1 ===
  Features: 115, Rows: train=207,162 valid=48,297
  Target: water_level

  --- CatBoost ---
0:	learn: 16.0424762	test: 16.0260053	best: 16.0260053 (0)	total: 252ms	remaining: 41m 56s
500:	learn: 0.0235099	test: 0.0338605	best: 0.0338605 (500)	total: 6.04s	remaining: 1m 54s
1000:	learn: 0.0155323	test: 0.0311612	best: 0.0311479 (984)	total: 12s	remaining: 1m 47s
1500:	learn: 0.0123470	test: 0.0299776	best: 0.0299729 (1498)	total: 17.7s	remaining: 1m 40s
2000:	learn: 0.0105477	test: 0.0294271	best: 0.0294268 (1998)	total: 23.4s	remaining: 1m 33s
2500:	learn: 0.0093859	test: 0.0292007	best: 0.0292005 (2499)	total: 29.1s	remaining: 1m 27s
3000:	learn: 0.0085929	test: 0.0290197	best: 0.0290197 (3000)	total: 35.2s	remaining: 1m 22s
3500:	learn: 0.0079848	test: 0.0289015	best: 0.0289011 (3492)	total: 41.2s	remaining: 1m 16s
4000:	learn: 0.0074882	test: 0.0288049	best: 0.0288020 (3944)	total: 47s	remaining: 1m 10s
4500:	learn: 0.0070858	test: 0.0287617	best: 0.0287

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 300 rounds
[300]	valid_0's rmse: 0.0299829
[600]	valid_0's rmse: 0.0280443
[900]	valid_0's rmse: 0.0273113
[1200]	valid_0's rmse: 0.0270202
[1500]	valid_0's rmse: 0.0268176
[1800]	valid_0's rmse: 0.0266787
[2100]	valid_0's rmse: 0.0265596
[2400]	valid_0's rmse: 0.0264545
[2700]	valid_0's rmse: 0.0263943
[3000]	valid_0's rmse: 0.0263276
[3300]	valid_0's rmse: 0.0262706
[3600]	valid_0's rmse: 0.0262489
[3900]	valid_0's rmse: 0.0262209
[4200]	valid_0's rmse: 0.026196
[4500]	valid_0's rmse: 0.0261804
[4800]	valid_0's rmse: 0.0261739
Early stopping, best iteration is:
[4617]	valid_0's rmse: 0.0261709
  LGB Raw RMSE: 0.026171, Std: 0.001551
  LGB best iter: 4617

=== Model 1, NodeType 2 ===
  Features: 102, Rows: train=45,283,176 valid=10,557,156
  Target: water_level

  --- CatBoost ---
0:	learn: 12.2688757	test: 12.2724987	best: 12.2724987 (0)	total: 317ms	remaining: 26m 25s
500:	learn: 0.0862295	test: 0.1102992	best: 0.1102992 (500)	total

In [14]:
# =========================================================

# ENSEMBLE WEIGHTS: Fixed 50/50 (v7)

# =========================================================

optimal_weights = {}


for (m, nt), (y_true, p_cb, p_lgb) in valid_preds.items():

    optimal_weights[(m, nt)] = 0.5

    sd = STD_DEV_DICT.get((m, nt), 1.0)

    p_ens = 0.5 * p_cb + 0.5 * p_lgb

    r = np.sqrt(np.mean((y_true - p_ens)**2))

    print(f"  M{m} NT{nt}: fixed 50/50, RMSE={r:.6f} (Std={r/sd:.6f})")


print(f"\nWeights: {optimal_weights}")

  M1 NT1: fixed 50/50, RMSE=0.023604 (Std=0.001399)
  M1 NT2: fixed 50/50, RMSE=0.065798 (Std=0.004576)
  M2 NT2: fixed 50/50, RMSE=0.117165 (Std=0.042963)

Weights: {(1, 1): 0.5, (1, 2): 0.5, (2, 2): 0.5}


In [15]:
# =========================================================

# LOCAL VALIDATION

# =========================================================

sol = pd.concat(sol_parts, ignore_index=True)


print("=== CatBoost only ===")

sub_cb = pd.concat(sub_parts_cb, ignore_index=True)

score_cb = local_score(sol, sub_cb)


print("\n=== LightGBM only ===")

sub_lgb = pd.concat(sub_parts_lgb, ignore_index=True)

score_lgb = local_score(sol, sub_lgb)


print("\n=== Ensemble (50/50) ===")

sub_ens_50 = sub_cb.copy()

sub_ens_50[TARGET_COL] = 0.5 * sub_cb[TARGET_COL].values + 0.5 * sub_lgb[TARGET_COL].values

score_ens_50 = local_score(sol, sub_ens_50)


print("\n=== Ensemble (optimal per-submodel weights) ===")

sub_ens_opt = sub_cb.copy()

for (m, nt), w in optimal_weights.items():

    mask = (sub_ens_opt['model_id'] == m) & (sub_ens_opt['node_type'] == nt)

    sub_ens_opt.loc[mask, TARGET_COL] = (

        w * sub_cb.loc[mask, TARGET_COL].values +

        (1 - w) * sub_lgb.loc[mask, TARGET_COL].values

    )

score_ens_opt = local_score(sol, sub_ens_opt)


print(f"\nCB: {score_cb:.6f}, LGB: {score_lgb:.6f}, ENS 50/50: {score_ens_50:.6f}, ENS opt: {score_ens_opt:.6f}")

print(f"v4 LB: 0.0319, v5 local (partial): ~0.047")

=== CatBoost only ===
  Model 1: Std RMSE = 0.001631 (15 events)
    NT1: raw RMSE mean=0.019969
    NT2: raw RMSE mean=0.029891
  Model 2: Std RMSE = 0.033719 (15 events)
    NT2: raw RMSE mean=0.091957
  Overall: 0.017675

=== LightGBM only ===
  Model 1: Std RMSE = 0.001602 (15 events)
    NT1: raw RMSE mean=0.017999
    NT2: raw RMSE mean=0.030749
  Model 2: Std RMSE = 0.035090 (15 events)
    NT2: raw RMSE mean=0.095695
  Overall: 0.018346

=== Ensemble (50/50) ===
  Model 1: Std RMSE = 0.001377 (15 events)
    NT1: raw RMSE mean=0.016192
    NT2: raw RMSE mean=0.025792
  Model 2: Std RMSE = 0.032151 (15 events)
    NT2: raw RMSE mean=0.087680
  Overall: 0.016764

=== Ensemble (optimal per-submodel weights) ===
  Model 1: Std RMSE = 0.001377 (15 events)
    NT1: raw RMSE mean=0.016192
    NT2: raw RMSE mean=0.025792
  Model 2: Std RMSE = 0.032151 (15 events)
    NT2: raw RMSE mean=0.087680
  Overall: 0.016764

CB: 0.017675, LGB: 0.018346, ENS 50/50: 0.016764, ENS opt: 0.016764
v4 

In [16]:
# =========================================================

# v7: SEED-BLEND REFIT + PREDICT (fused per submodel)

# =========================================================

print("=== Seed-Blend Refit + Predict ===")

print(f"Seeds: {SEED_LIST}")

print(f"Skip: {SKIP_SUBMODELS}")


# FREE MEMORY: delete validation models + caches BEFORE refit

del cb_models, lgb_models

del sol_parts, sub_parts_cb, sub_parts_lgb

try:

    del sol, sub_cb, sub_lgb, sub_ens_50, sub_ens_opt

except NameError:

    pass

del valid_preds

EVENT_FRAME_CACHE.clear()

RAIN_CACHE.clear()

WARMUP_CACHE.clear()

WARMUP_DYN_CACHE.clear()

TIMESTEP_DURATION_CACHE.clear()

_free_memory()

print("  Freed validation data & caches")


# Load sample submission keys

sample_path = Path('/kaggle/input/notebooks/johndoe2011/still-naive-2-0/submission.csv')

if not sample_path.exists():

    for p in [Path('./heuristic_rain_analog_submission.parquet'),

              Path('/kaggle/input/flood-comp-dataset/sample_submission.parquet'),

              Path('/kaggle/input/urban-flood-modelling/sample_submission.parquet')]:

        if p.exists(): sample_path = p; break


print(f"Using sample: {sample_path}")

sample = read_table(sample_path)

sample_keys = sample[['row_id', 'model_id', 'event_id', 'node_type', 'node_id']].copy()

sample_keys = sample_keys.sort_values('row_id').reset_index(drop=True)

del sample; gc.collect()


def add_seq(df):

    out = df.sort_values(['node_id', 'timestep']).reset_index(drop=True)

    out['seq'] = out.groupby('node_id').cumcount().astype('int64')

    return out


def get_event_keys(sk, m, e, nt):

    ek = sk[(sk['model_id']==m) & (sk['event_id']==e) & (sk['node_type']==nt)].copy()

    if ek.empty: return ek

    ek = ek.sort_values('row_id').reset_index(drop=True)

    ek['seq'] = ek.groupby(['model_id','event_id','node_type','node_id']).cumcount().astype('int64')

    return ek


all_test_parts = []


for m in sorted(train_events):

    for nt in (1, 2):

        if (m, nt) in SKIP_SUBMODELS:

            print(f"\n  Skipping M{m} NT{nt} (optimized separately)")

            continue


        print(f"\n{'='*60}")

        print(f"SEED BLEND: model_id={m}, node_type={nt}")

        print(f"{'='*60}")


        # Phase 1: Build training data (ALL events)

        EVENT_FRAME_CACHE.clear()

        RAIN_CACHE.clear()

        WARMUP_CACHE.clear()

        WARMUP_DYN_CACHE.clear()

        TIMESTEP_DURATION_CACHE.clear()

        _free_memory()


        all_frames = [build_supervised_event_frame(m, 'train', ed, nt)

                      for ed in train_events[m]]


        # Clear caches after building (data is in all_frames list)

        EVENT_FRAME_CACHE.clear()

        WARMUP_CACHE.clear()

        WARMUP_DYN_CACHE.clear()

        _free_memory()


        all_df = pd.concat(all_frames, ignore_index=True)

        del all_frames; gc.collect()


        feat, cat = get_features(all_df)

        feat_lgb = [f for f in feat if f != 'node_id_cat']

        target = TARGET_COL


        print(f"  Features: {len(feat)}, Rows: {len(all_df):,}")


        # Determine refit iterations from validation specs

        sp = specs.get((m, nt))

        if sp:

            cb_refit_iter = min(int(sp['best_iter_cb'] * 1.3) + 100,

                               CB_PARAMS[(m, nt)]['iterations'])

            lgb_refit_iter = min(int(sp['best_iter_lgb'] * 1.3) + 100,

                                LGB_PARAMS[(m, nt)]['n_estimators'])

        else:

            cb_refit_iter = CB_PARAMS[(m, nt)]['iterations']

            lgb_refit_iter = LGB_PARAMS[(m, nt)]['n_estimators']


        print(f"  CB refit iters: {cb_refit_iter}, LGB refit iters: {lgb_refit_iter}")


        # Phase 2: Train all seeds

        seed_models = []

        for seed in SEED_LIST:

            t0 = time.time()


            cb_p = {**CB_PARAMS[(m, nt)], 'random_seed': seed,

                    'iterations': cb_refit_iter}

            cb = CatBoostRegressor(**cb_p)

            cb.fit(all_df[feat], all_df[target], cat_features=cat,

                   verbose=cb_p.get('verbose', 500))


            lgb_p = {**LGB_PARAMS[(m, nt)], 'random_state': seed,

                     'n_estimators': lgb_refit_iter}

            lgb_m = lgb.LGBMRegressor(**lgb_p)

            lgb_m.fit(all_df[feat_lgb], all_df[target])


            seed_models.append((cb, lgb_m))

            elapsed = time.time() - t0

            print(f"    Seed {seed} trained ({elapsed:.0f}s)")


        del all_df; gc.collect()

        EVENT_FRAME_CACHE.clear()

        RAIN_CACHE.clear()

        WARMUP_CACHE.clear()

        WARMUP_DYN_CACHE.clear()

        TIMESTEP_DURATION_CACHE.clear()

        _free_memory()


        # Phase 3: Predict test events

        n_models = len(seed_models) * 2  # CB + LGB per seed

        print(f"\n  Predicting with {n_models} models (test events)...")


        for ed in test_events[m]:

            e = eid(ed)

            print(f"    M{m} E{e} NT{nt}...", end=' ')

            t_start = time.time()


            test_df = build_supervised_event_frame(

                m, 'test', ed, nt, include_target=False)


            # Ensure all expected features exist

            for fc in feat:

                if fc not in test_df.columns:

                    test_df[fc] = 0


            # Average predictions across all seeds (CB + LGB)

            preds_sum = np.zeros(len(test_df), dtype='float64')

            for cb_s, lgb_s in seed_models:

                preds_sum += cb_s.predict(test_df[feat])

                preds_sum += lgb_s.predict(test_df[feat_lgb])

            test_df[TARGET_COL] = (preds_sum / n_models).astype('float64')


            test_df = add_seq(test_df)

            ek = get_event_keys(sample_keys, m, e, nt)

            if ek.empty:

                print("SKIP"); continue

            aligned = ek.merge(test_df[KEY_COLS + ['seq', TARGET_COL]],

                              on=KEY_COLS + ['seq'], how='left')

            missing = aligned[TARGET_COL].isna().sum()

            elapsed = time.time() - t_start

            if missing > 0:

                print(f"WARN:{missing} ({elapsed:.1f}s)")

                aligned[TARGET_COL] = aligned[TARGET_COL].fillna(0)

            else:

                print(f"OK({len(aligned):,}, {elapsed:.1f}s)")

            all_test_parts.append(aligned[['row_id'] + KEY_COLS + [TARGET_COL]])


            del test_df

            _free_memory()


        # Phase 4: Cleanup — delete all models for this submodel

        del seed_models

        EVENT_FRAME_CACHE.clear()

        RAIN_CACHE.clear()

        WARMUP_CACHE.clear()

        WARMUP_DYN_CACHE.clear()

        TIMESTEP_DURATION_CACHE.clear()

        _free_memory()

        print(f"  M{m} NT{nt} done, models freed.")


print("\nAll seed-blend predictions complete.")

=== Seed-Blend Refit + Predict ===
Seeds: [56, 228, 322]
Skip: {(2, 1)}
  Freed validation data & caches
Using sample: /kaggle/input/notebooks/johndoe2011/still-naive-2-0/submission.csv

SEED BLEND: model_id=1, node_type=1
  Features: 115, Rows: 255,459
  CB refit iters: 10000, LGB refit iters: 6102
0:	learn: 16.0394645	total: 11.7ms	remaining: 1m 56s
500:	learn: 0.0235869	total: 5.68s	remaining: 1m 47s
1000:	learn: 0.0159692	total: 11.4s	remaining: 1m 42s
1500:	learn: 0.0127389	total: 17.1s	remaining: 1m 36s
2000:	learn: 0.0109467	total: 22.8s	remaining: 1m 31s
2500:	learn: 0.0097929	total: 28.5s	remaining: 1m 25s
3000:	learn: 0.0089751	total: 34.3s	remaining: 1m 19s
3500:	learn: 0.0083594	total: 40.1s	remaining: 1m 14s
4000:	learn: 0.0078669	total: 45.8s	remaining: 1m 8s
4500:	learn: 0.0074632	total: 51.6s	remaining: 1m 3s
5000:	learn: 0.0071084	total: 57.3s	remaining: 57.3s
5500:	learn: 0.0067964	total: 1m 3s	remaining: 51.6s
6000:	learn: 0.0065415	total: 1m 8s	remaining: 45.9s
6500

In [17]:
m2nt1_source = None

print("WARNING: No existing submission found for M2 NT1 placeholder, using zeros")

m2nt1_keys = sample_keys[

    (sample_keys['model_id'] == 2) & (sample_keys['node_type'] == 1)

].copy()

m2nt1_keys[TARGET_COL] = 0.0

all_test_parts.append(m2nt1_keys[['row_id'] + KEY_COLS + [TARGET_COL]])

print(f"  M2 NT1 zeros: {len(m2nt1_keys):,} rows")

  M2 NT1 zeros: 1,422,036 rows


In [18]:
# =========================================================
# ASSEMBLE & SAVE SUBMISSION
# =========================================================

test_sub = pd.concat(all_test_parts, ignore_index=True)
test_sub = test_sub.sort_values('row_id').reset_index(drop=True)
test_sub = test_sub[SUBMISSION_COLS]

print(f"\nSubmission: {len(test_sub):,} rows (expected {len(sample_keys):,})")

assert len(test_sub) == len(sample_keys), f"Row count mismatch: got {len(test_sub)}, expected {len(sample_keys)}"
assert test_sub['water_level'].isna().sum() == 0, "NaN values in submission!"
assert np.isfinite(test_sub['water_level']).all(), "Non-finite values in submission!"

# Check per-submodel row counts
for m in [1, 2]:
    for nt in [1, 2]:
        n = ((test_sub['model_id'] == m) & (test_sub['node_type'] == nt)).sum()
        n_exp = ((sample_keys['model_id'] == m) & (sample_keys['node_type'] == nt)).sum()
        status = "OK" if n == n_exp else "MISMATCH"
        print(f"  M{m} NT{nt}: {n:,} rows ({status}, expected {n_exp:,})")

out_path = Path('submission_v7_seedblend.parquet')
test_sub.to_parquet(out_path, index=False)
print(f"\nSaved: {out_path} ({len(test_sub):,} rows)")


Submission: 50,910,192 rows (expected 50,910,192)
  M1 NT1: 84,762 rows (OK, expected 84,762)
  M1 NT2: 18,527,976 rows (OK, expected 18,527,976)
  M2 NT1: 1,422,036 rows (OK, expected 1,422,036)
  M2 NT2: 30,875,418 rows (OK, expected 30,875,418)

Saved: submission_v7_seedblend.parquet (50,910,192 rows)
